In [1]:
import os
import pandas as pd
from astropy.time import Time
import matplotlib.pyplot as plt

from datetime import datetime, timedelta
import numpy as np

from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components

In [2]:
def filter_intervals(gps_start, gps_end, intervals):
    intervals_crop = intervals[(intervals['int2'] >= gps_start) & (intervals['int1'] <= gps_end)].sort_values('int1')
    intervals_crop['int1'] = intervals_crop['int1'].astype('float64')
    
    total_triggers_winddow = 0
    filtered_chunks = []
    
    for chunk in pd.read_csv(basedata + '/triggers_total.csv', chunksize=10000000):
    
        chunk['time'] = chunk['time'].astype('float64')
        
        chunk = chunk[(chunk['time'] >= gps_start) & (chunk['time'] <= gps_end)].sort_values('time')
        if chunk.empty:
            continue
    
        total_triggers_winddow += len(chunk)
        
        merged = pd.merge_asof(chunk, intervals_crop, left_on='time', right_on='int1', direction='backward')
        valid = merged[(merged['time'] >= merged['int1']) & (merged['time'] <= merged['int2'])]
        
        filtered_chunks.append(valid[['time', 'frequency', 'snr']])
    
    triggers_crop = pd.concat(filtered_chunks, ignore_index=True)

    return triggers_crop

In [3]:
def clusterize_triggers1D(gltab, window, min_triggers):
    
    if len(gltab) == 0:
        return pd.DataFrame()

    table = gltab.copy()
    table = table.sort_values(by='time').reset_index(drop=True)

    times = table['time'].values
    deltas = times[1:] - times[:-1]

    new_cluster_mask = np.concatenate([[False], deltas > window])
    cluster_ids = np.cumsum(new_cluster_mask)

    table['cluster'] = cluster_ids

    # filter: Retains only clusters with "min_triggers" or more triggers
    counts = table['cluster'].value_counts()
    valid_clusters = counts[counts >= min_triggers].index
    
    table_filtered = table[table['cluster'].isin(valid_clusters)]

    if table_filtered.empty:
        return pd.DataFrame()
    
    # groups by cluster and extracts the representative (row with the highest SNR) + extremes
    summary_rows = []

    print(table_filtered)
    
    for cl, group in table_filtered.groupby('cluster'):
        # Encontra o trigger com maior SNR dentro deste grupo
        max_row = group.loc[group['snr'].idxmax()]

        summary_rows.append({
            'cluster': cl,
            'num_triggers': len(group), # quantidade de triggers no cluster
            'tstart': group['time'].min(),
            'tend': group['time'].max(),
            'time': max_row['time'],
            'frequency': max_row['frequency'],
            'min_freq': group['frequency'].min(),
            'max_freq': group['frequency'].max(),
            'snr': max_row['snr']
        })

    return pd.DataFrame(summary_rows)

In [4]:
def clusterize_triggers2D(gltab, window_T, window_F, min_triggers):

    if len(gltab) == 0:
        return pd.DataFrame()

    table = gltab.copy()

    times = table['time'].values
    freqs = table['frequency'].values
    n = len(times)

    # Ordena por tempo para facilitar a janela deslizante
    sort_idx = np.argsort(times)
    times_sorted = times[sort_idx]
    freqs_sorted = freqs[sort_idx]
    
    connections = []

    # Para cada trigger, olha APENAS para os próximos que estão dentro da janela de tempo
    for i in range(n):
        t_curr = times_sorted[i]
        f_curr = freqs_sorted[i]

        # Busca apenas os candidatos dentro da janela de tempo t_curr + window_T
        for j in range(i + 1, n):
            if times_sorted[j] - t_curr > window_T:
                break  # Como está ordenado, se estourou o tempo, para a busca imediatamente!

            # Testa a condição de frequência
            if abs(freqs_sorted[j] - f_curr) /f_curr <= window_F:
                # Conecta o trigger i ao trigger j
                connections.append((sort_idx[i], sort_idx[j]))
    
    # Monta os grupos com base nas conexões encontradas
    if connections:
        pairs = np.array(connections)
        data = np.ones(len(pairs), dtype=bool)
        adj = csr_matrix((data, (pairs[:, 0], pairs[:, 1])), shape=(n, n))
        _, cluster_ids = connected_components(csgraph=adj, directed=False)
    else:
        cluster_ids = np.arange(n)

    table['cluster'] = cluster_ids

    # filter: Retains only clusters with "min_triggers" or more triggers
    counts = table['cluster'].value_counts()
    valid_clusters = counts[counts >= min_triggers].index
    
    table_filtered = table[table['cluster'].isin(valid_clusters)]

    if table_filtered.empty:
        return pd.DataFrame()
    
    # groups by cluster and extracts the representative (row with the highest SNR) + extremes
    summary_rows = []
    
    for cl, group in table_filtered.groupby('cluster'):
        # Encontra o trigger com maior SNR dentro deste grupo
        max_row = group.loc[group['snr'].idxmax()]

        summary_rows.append({
            'cluster': cl,
            'num_triggers': len(group), # quantidade de triggers no cluster
            'tstart': group['time'].min(),
            'tend': group['time'].max(),
            'time': max_row['time'],
            'frequency': max_row['frequency'],
            'min_freq': group['frequency'].min(),
            'max_freq': group['frequency'].max(),
            'snr': max_row['snr']
        })

    return pd.DataFrame(summary_rows)

In [5]:
def run_routine(current_day, end_day, intervals, min_triggers, global_cluster_ID=0):

    gps_start_filt = Time(current_day, scale="utc").gps
    gps_end_filt = Time(end_day, scale="utc").gps

    triggers_crop = filter_intervals(gps_start_filt, gps_end_filt, intervals)
    
    while current_day <= end_day:
        
        # declare the start and end of the current day
        day_start = datetime(current_day.year, current_day.month, current_day.day, 0, 0, 0)
        day_end   = datetime(current_day.year, current_day.month, current_day.day, 23, 59, 59)
    
        print(f"Processing day: {current_day.date()}")
    
        # convert the time to gpsTIME
        gps_start = Time(day_start, scale="utc").gps
        gps_end   = Time(day_end, scale="utc").gps
    
        # limit the intervals to the current day only
        day_interval = intervals[(intervals['int2'] >= gps_start) & (intervals['int1'] <= gps_end)].copy()
    
        # cut intervals before gps_start and after gps_end
        day_interval['start_recortado'] = day_interval['int1'].clip(lower=gps_start)
        day_interval['end_recortado']   = day_interval['int2'].clip(upper=gps_end)
    
        # duration of the intervals of the current day
        day_duration_sec = (day_interval['end_recortado'] - day_interval['start_recortado']).sum()
        day_duration_hrs = day_duration_sec / 3600
    
        print(f'Opening hours for this day: {day_duration_hrs:.2f}')
    
        # triggers of the current day
        trigfiles = triggers_crop[(triggers_crop['time'] >= gps_start) & (triggers_crop['time'] <= gps_end)]
        gltab = trigfiles.copy()
    
        trigger_count.append(len(gltab))
        print(f'Triggers count for this day: {len(gltab)}\n')
    
        # CLUSTERING OF THE CURRENT DAY
        if not gltab.empty:
            gltab = gltab.sort_values(by="time").drop_duplicates(subset=["time"]).reset_index(drop=True)
    
            # the MAIN FUNCTION of this clustering process
            # clustered = clusterize_triggers1D(gltab, window_T, min_triggers)
            clustered = clusterize_triggers2D(gltab, window_T, window_F, min_triggers)
    
            glitch_count.append(len(clustered))
            print(f'\nGlitches count for this day: {len(clustered)}\n')
            
            # adds the day's result to the list if there are glitches
            if not clustered.empty:
                
                # adjusts the day's IDs by adding the accumulated ID
                clustered['cluster'] += global_cluster_ID
    
                # update the ID for the next day to avoid repeating numbers
                global_cluster_ID = clustered['cluster'].max() + 1
                
                clustered_list.append(clustered)
                
        current_day += timedelta(days=1)

    # appends to the FINAL DataFrame every day
    if clustered_list:
        df_glitches_final = pd.concat(clustered_list, ignore_index=True)
    else:
        df_glitches_final = pd.DataFrame()

    return df_glitches_final

///////////////////////////////////////////////////////////////////

In [6]:
start_day_O3a = datetime(2019, 4, 1, 15, 0, 0)
end_day_O3a   = datetime(2019, 10, 1, 15, 0, 0)

start_day_O3b = datetime(2019, 11, 1, 0, 0, 0)
end_day_O3b   = datetime(2020, 3, 27, 23, 59, 59)

end_day_test = datetime(2019, 4, 10, 15, 0, 0)

In [7]:
snr_thresholds = [6.5, 10]         # corte de SNR
window_T, window_F = 0.1, 20       # janela de clusterização (segundos)
min_triggers = 30

clust_dim, min_trigg = '2D', f'cut{min_triggers}'

glitch_par = f'{clust_dim}{min_trigg}'
glitch_par = f'{clust_dim}{min_trigg}_normFreq'

In [8]:
gps_start = Time(start_day_O3a, scale="utc").gps
gps_end = Time(end_day_O3a, scale="utc").gps

gps_end_test = Time(end_day_test, scale="utc").gps

In [9]:
basedata = os.getcwd()
intervals = pd.read_csv(basedata + '/zrepository/gwoscO3atxt.txt', sep=' ', header=None, names=['int1', 'int2', 'duration'])

In [10]:
trigger_count = []
glitch_count = []

clustered_list = []

df_glitches_final = run_routine(start_day_O3a, end_day_O3a, intervals, min_triggers)

Processing day: 2019-04-01
Opening hours for this day: 9.00
Triggers count for this day: 212320


Glitches count for this day: 43

Processing day: 2019-04-02
Opening hours for this day: 15.39
Triggers count for this day: 172113


Glitches count for this day: 57

Processing day: 2019-04-03
Opening hours for this day: 23.27
Triggers count for this day: 267901


Glitches count for this day: 128

Processing day: 2019-04-04
Opening hours for this day: 20.03
Triggers count for this day: 929860


Glitches count for this day: 2958

Processing day: 2019-04-05
Opening hours for this day: 23.29
Triggers count for this day: 491367


Glitches count for this day: 1276

Processing day: 2019-04-06
Opening hours for this day: 21.30
Triggers count for this day: 414648


Glitches count for this day: 111

Processing day: 2019-04-07
Opening hours for this day: 24.00
Triggers count for this day: 352671


Glitches count for this day: 370

Processing day: 2019-04-08
Opening hours for this day: 23.99
Triggers 

In [11]:
df_glitches_final

,cluster,num_triggers,tstart,tend,time,frequency,min_freq,max_freq,snr
0,128,42,1.238166e+09,1.238166e+09,1.238166e+09,24.171913,18.819545,1426.656564,7.195065
1,229,38,1.238167e+09,1.238167e+09,1.238167e+09,215.134078,75.737187,259.684170,6.404389
2,454,30,1.238167e+09,1.238167e+09,1.238167e+09,21.824261,17.731117,60.831633,8.541214
3,500,36,1.238167e+09,1.238167e+09,1.238167e+09,1270.136150,21.775510,1755.427748,7.067101
4,748,33,1.238168e+09,1.238168e+09,1.238168e+09,58.457837,56.940496,59.310304,7.124077
...,...,...,...,...,...,...,...,...,...
80493,3284433,31,1.253977e+09,1.253977e+09,1.253977e+09,23.091503,20.626565,1433.373222,7.162734
80494,3284445,786,1.253977e+09,1.253977e+09,1.253977e+09,1280.344030,17.731117,1942.651041,95.633463
80495,3284466,30,1.253977e+09,1.253977e+09,1.253977e+09,20.025103,17.731117,26.742411,8.553306
80496,3284470,39,1.253977e+09,1.253977e+09,1.253977e+09,22.212176,16.889483,32.842241,7.467254


In [12]:
glitch_par

'2Dcut30'

In [13]:
here = os.getcwd()
save_path = here + '/zrepository/gdchar/O3a_normFreq/'

In [14]:
df_glitches_final.to_csv(save_path + '/gdcharO3a_' + glitch_par + '.csv', index=False)

calcular os intervals da O3a para ver o glitch per hour V

In [20]:
O3a_intervals = pd.read_csv(here + '/zrepository/gwoscO3atxt.txt', names=['start', 'end', 'duration'], header=None, sep=' ')
allIntervalO3a = O3a_intervals['duration'].values.sum() / 3600
allIntervalO3a

np.float64(3344.1469444444447)

In [21]:
glitches = pd.read_csv(here + '/zrepository/gdchar/O3a_normFreq/gdcharO3a_2Dcut30_normFreq.csv')
len_glitches = len(glitches)

In [22]:
O3a_glitches_per_hour = len_glitches / allIntervalO3a
O3a_glitches_per_hour

np.float64(24.071310662269042)